original notebook is in /gpfs01/berens/user/inwabufo/time_distance/time_distance/t_cross_long/analyse_data.ipynb

In [1]:
%load_ext autoreload
%autoreload 2
%env CUDA_VISIBLE_DEVICES=0

env: CUDA_VISIBLE_DEVICES=0


In [3]:
import os
import numpy as np
import pandas as pd


In [4]:
root = '/gpfs01/berens/user/inwabufo/time_distance/time_distance'

In [5]:
df  = pd.read_csv(os.path.join(root, 'datasets/NAKO/diseased_eyes.csv'))
df.shape

(5102, 8)

In [6]:
disease_labels = {
'd_an_aug_1': 'cataract', 
'd_an_aug_2': 'glaucoma', 
'd_an_aug_3': 'MD',
'd_an_metdm2c_2': 'blindness', 
'd_an_metdm2c_1': 'retinopathy',
'd_an_met_1': 'diabetes_mellitus'
                    }

In [7]:
df.head()

,ID,basis_sex,d_an_aug_1,d_an_aug_2,d_an_aug_3,d_an_metdm2c_2,d_an_metdm2c_1,d_an_met_1
0,100049,Female,No,No,No,No,No,Yes
1,100069,Male,No,No,No,No,No,Yes
2,100082,Male,Yes,No,No,Missing,Missing,No
3,100091,Female,No,Yes,No,Missing,Missing,No
4,100094,Female,Yes,No,No,Missing,Missing,No


In [8]:
df.rename(columns=disease_labels, inplace=True)

In [9]:
df.head()

,ID,basis_sex,cataract,glaucoma,MD,blindness,retinopathy,diabetes_mellitus
0,100049,Female,No,No,No,No,No,Yes
1,100069,Male,No,No,No,No,No,Yes
2,100082,Male,Yes,No,No,Missing,Missing,No
3,100091,Female,No,Yes,No,Missing,Missing,No
4,100094,Female,Yes,No,No,Missing,Missing,No


In [29]:
df.shape

(5102, 8)

In [28]:
df.to_csv('/gpfs01/berens/user/inwabufo/leverage/leverage-data/metadata/nako_eye_disease.csv', index = False)

In [17]:
dict1 = {}
for col in df.columns[2:]:
    if col not in dict1.keys():
        dict1[col] = []
        dict1[col] =df[col].value_counts().to_dict()
dict1

{'cataract': {'No': 3039, 'Yes': 2042, 'Missing': 21},
 'glaucoma': {'No': 4102, 'Yes': 971, 'Missing': 29},
 'MD': {'No': 4627, 'Yes': 429, 'Missing': 46},
 'blindness': {'Missing': 2728, 'No': 2373, 'Yes': 1},
 'retinopathy': {'Missing': 2771, 'No': 2240, 'Yes': 91},
 'diabetes_mellitus': {'No': 2716, 'Yes': 2380, 'Missing': 6}}

In [ ]:
k = pd.DataFrame(dict1).T.reset_index()
k
table = pd.pivot_table(
    k, values=['No', 'Yes', 'Missing'], index=["index"],  aggfunc="sum"
)
table
# Row-wise: what % of each condition's total is "Yes"
k['Yes_pct_row'] = k['Yes'] / k[['No', 'Yes', 'Missing']].sum(axis=1) * 100

# Column-wise: what % of the overall "Yes" total does each condition contribute
k['Yes_pct_col'] = k['Yes'] / k['Yes'].sum() * 100
k[['Yes_pct_row', 'Yes_pct_col']] = k[['Yes_pct_row', 'Yes_pct_col']].round(1)
k

,index,No,Yes,Missing
0,cataract,3039,2042,21
1,glaucoma,4102,971,29
2,MD,4627,429,46
3,blindness,2373,1,2728
4,retinopathy,2240,91,2771
5,diabetes_mellitus,2716,2380,6


In [6]:
def change_lbls(dataset_, diseased_ids):
    for i, ids in enumerate(dataset_.ids):
        if ids in diseased_ids:
            dataset_.labels[i] = 1
        else:
            dataset_.labels[i] = 0

In [ ]:
img_size = 224
batch_size = 64


all_data = load_nako(image_size =  img_size, 
                batch_size = batch_size, 
                label_name = None, 
                return_loader = False, 
                normalize = True, 
                return_all = True, 
                return_ids = False)
all_dataset_train, _, _, _, transforms_ = all_data
len(all_dataset_train)
diseased_ids = list(df['ID'])
change_lbls(all_dataset_train, diseased_ids)
all_dataset_train.labels = all_dataset_train.labels.astype(int)
print(np.unique(all_dataset_train.labels, return_counts = True))
